In [3]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

# Load environment variables from your .env file
load_dotenv()

# Verify your Groq key is loaded properly
if os.getenv('GROQ_API_KEY'):
    print("Groq API Key loaded successfully")

Groq API Key loaded successfully


In [4]:
# Initialize ChatGroq dynamically using .env variables
llm = ChatGroq(
    model=os.getenv('GROQ_MODEL', 'llama3-8b-8192'),  # Uses .env value, defaults to llama3 if missing
    temperature=0.0,
    groq_api_key=os.getenv('GROQ_API_KEY')
)

print(f"Groq LLM initialized successfully with model: {llm.model}")

Groq LLM initialized successfully with model: openai/gpt-oss-120b


In [6]:
# # You can now invoke it exactly the same way as shown in the video!
# response = llm.invoke("tell me something about Subhas Chandra Boss")
# print(response.content)

**Subhas Chandra Bose (1897‑1945)**  

- **Early life & education**  
  - Born on 23 January 1897 in Cuttack, Odisha (then part of British India).  
  - Came from a well‑educated Bengali family; his father, Janakinath Bose, was a lawyer.  
  - Attended the University of Calcutta, where he earned a degree in philosophy, and later went to England to prepare for the Indian Civil Service (ICS). He passed the rigorous exams but resigned from the ICS in 1921 to join the freedom‑movement full‑time.

- **Political career**  
  - Joined the Indian National Congress (INC) and quickly rose through its ranks, becoming a prominent leader in the 1920s and 1930s.  
  - Known for his charismatic oratory, strong organizational skills, and a more militant stance toward British rule compared with the non‑violent approach championed by Mahatma Gandhi.  
  - Served as President of the Indian National Congress twice (1938 and 1939). He resigned the second time after a disagreement over the Congress’s policy

## **RAG Implementation with your own text data**

### **STEP 1 : Preparing Document for your Text**

In [5]:
from langchain_core.documents import Document
my_text = """Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.[1]

High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g., text, images, audio, and videos).

The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach these goals, AI researchers use techniques including state space search and mathematical optimisation, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics.[b] AI also draws upon psychology, linguistics, philosophy, neuroscience, and other fields.[2] Some companies, such as OpenAI, Google DeepMind, and Meta, aim to create artificial general intelligence (AGI)—AI that can complete nearly any cognitive task at least as well as a human.[3]

Artificial intelligence was founded as an academic discipline in 1956.[4] The field went through multiple cycles of optimism throughout its history,[5][6] followed by periods of disappointment and loss of funding, known as AI winters.[7][8] Funding and interest increased substantially after 2012, when graphics processing units (GPUs) started being used to accelerate neural networks, and deep learning outperformed previous AI techniques.[9] This growth accelerated further after 2017 with the transformer architecture.[10] The 2020s AI boom was spurred on largely by advances in generative AI, which can create and modify media. In addition to AI safety and unintended consequences and harms from the use of AI, ethical concerns, AI's long-term effects, environmental effects,[11] and potential existential risks have prompted discussions of AI regulation. 
"""

docs = [Document(page_content=my_text,metadata = {"source" : "ABC","documentID" : "doc1"})]
docs


[Document(metadata={'source': 'ABC', 'documentID': 'doc1'}, page_content="Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.[1]\n\nHigh-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g., text, images, audio, and videos).\n\nThe traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a

### **STEP 2 : Splitting the document into CHUNKS**

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50)

chunks = splitter.split_documents(docs)
chunks

[Document(metadata={'source': 'ABC', 'documentID': 'doc1'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.[1]'),
 Document(metadata={'source': 'ABC', 'documentID': 'doc1'}, page_content='High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g., text, images, audio, and videos).'),
 Document(metadata={'source': 'ABC', 'documentID': 'doc1'}, page_content='The traditional goals of AI resea

### **STEP 3 : Creating Embeddingd for the CHUNKS**

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings

C:\Sourav\Study\FDE\Study\RAG\RAG_MasterClass\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2478.02it/s]


HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [8]:
embeddings.embed_query("what is AI?")

[-0.024964412674307823,
 -0.009133641608059406,
 -0.0074615562334656715,
 0.01500905305147171,
 0.013310457579791546,
 -0.010036010295152664,
 0.07456009835004807,
 0.04267147555947304,
 0.016988525167107582,
 0.055951010435819626,
 -0.029681872576475143,
 -0.004356063902378082,
 0.02053230255842209,
 -0.04828272759914398,
 -0.058669451624155045,
 0.0423620231449604,
 -0.018919453024864197,
 -0.05299250781536102,
 -0.08717180043458939,
 -0.06998187303543091,
 -0.008720222860574722,
 0.0196782685816288,
 -0.04860039800405502,
 -0.04870658367872238,
 -0.03246690705418587,
 0.09295572340488434,
 0.004014179110527039,
 -0.06721635162830353,
 -0.0021876385435462,
 -0.011162926442921162,
 0.01210771407932043,
 -0.023686647415161133,
 0.10230045020580292,
 0.019198501482605934,
 -0.06650935858488083,
 0.041288379579782486,
 -0.040980394929647446,
 -0.027531713247299194,
 0.06851062178611755,
 -0.029644500464200974,
 -0.019207919016480446,
 -0.05420264974236488,
 0.016852661967277527,
 -0.0719

### **STEP 4 : Create and Store Embeddings in the Vector Store**

In [9]:
from langchain_community.vectorstores import Chroma
vectorstore = Chroma.from_documents(chunks, embeddings)
vectorstore

### **STEP 5 : Semantic Search**

In [11]:
vectorstore.similarity_search("what is AI?", k=3)

[Document(metadata={'documentID': 'doc1', 'source': 'ABC'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.[1]'),
 Document(metadata={'documentID': 'doc1', 'source': 'ABC'}, page_content='philosophy, neuroscience, and other fields.[2] Some companies, such as OpenAI, Google DeepMind, and Meta, aim to create artificial general intelligence (AGI)—AI that can complete nearly any cognitive task at least as well as a human.[3]'),
 Document(metadata={'source': 'ABC', 'documentID': 'doc1'}, page_content='The traditional goals of AI research i

### **Talk to LLM**

In [16]:
context = vectorstore.similarity_search("what is AI?", k=3)
context

[Document(metadata={'source': 'ABC', 'documentID': 'doc1'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.[1]'),
 Document(metadata={'documentID': 'doc1', 'source': 'ABC'}, page_content='philosophy, neuroscience, and other fields.[2] Some companies, such as OpenAI, Google DeepMind, and Meta, aim to create artificial general intelligence (AGI)—AI that can complete nearly any cognitive task at least as well as a human.[3]'),
 Document(metadata={'source': 'ABC', 'documentID': 'doc1'}, page_content='The traditional goals of AI research i

In [17]:
response = llm.invoke(f"what is AI? you can answer using following context: {context}") 
response.content

'Artificial Intelligence (AI) is a branch of computer science focused on creating systems that can perform tasks that normally require human intelligence. These tasks include learning, reasoning, problem‑solving, perception, language understanding, and even creativity. In practice, AI systems use algorithms—often based on statistical models, neural networks, or other forms of machine learning—to recognize patterns in data, make predictions, and take actions based on those insights.\n\n### Core Concepts\n\n| Concept | What It Means |\n|---------|----------------|\n| **Machine Learning (ML)** | Algorithms that improve automatically through experience (e.g., training on data). |\n| **Deep Learning** | A subset of ML that uses multi‑layered neural networks to model complex patterns (e.g., image and speech recognition). |\n| **Natural Language Processing (NLP)** | Techniques for understanding and generating human language (e.g., chatbots, translation). |\n| **Computer Vision** | Methods for